## FAQAgent 사용 예시입니다.

In [1]:
from src.agents.faq_agent import FAQAgent
from src.utils import get_chroma_db_client, get_openai_client
from src.db.chroma_db import get_or_create_collection
from src.config import cfg_engine

# llm client
llm = get_openai_client()

# DB client
client = get_chroma_db_client()
collection = get_or_create_collection(
    db_client=client, collection_name=cfg_engine.chroma_db.collection_name
)

2025-08-16 14:38:47,811 - chromadb.telemetry.product.posthog - INFO - Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
2025-08-16 14:38:47,828 - src.db.chroma_db - INFO - [ChromaDB] 'smartstore_faq' exists. Connect Collection


In [2]:
# Agent
agent = FAQAgent(llm=llm, db=collection)
agent.initialize(session_id="test1", chat_id="test1")
agent.delete_state()
agent.initialize(session_id="test1", chat_id="test1")

2025-08-16 14:38:47,837 - root - INFO - [ChatHistory] Connect collection: chat_history
2025-08-16 14:38:47,864 - root - INFO - [ChatHistory] Cleared messages for session_id=test1, chat_id=test1 from ChromaDB
2025-08-16 14:38:47,865 - root - INFO - [ChatHistory] Connect collection: chat_history


### Invoke

In [3]:
result = agent.invoke(input="안녕? 내 이름은 홍길동 이야.")

2025-08-16 14:36:36,447 - root - INFO - [Agent] Invoking...
2025-08-16 14:36:36,447 - root - INFO - [Agent] User: 안녕? 내 이름은 홍길동 이야.
2025-08-16 14:36:36,448 - root - INFO - [Agent] Calling LLM...
2025-08-16 14:36:40,490 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-08-16 14:36:40,500 - root - INFO - [Agent] Assistant:
안녕하세요 홍길동님, 만나서 반갑습니다! 스마트스토어 관련해 궁금한 점 있으신가요? 도와드릴 내용 알려주시면 답변해드릴게요.
2025-08-16 14:36:40,698 - root - INFO - [ChatHistory] Added 2 messages
2025-08-16 14:36:40,699 - root - INFO - Execution time of invoke(): 0m 4.25s


In [4]:
result = agent.invoke(input="내 이름이 뭐야?")

2025-08-16 14:36:40,702 - root - INFO - [Agent] Invoking...
2025-08-16 14:36:40,703 - root - INFO - [Agent] User: 내 이름이 뭐야?
2025-08-16 14:36:40,704 - root - INFO - [ChatHistory] Retrieved 2 messages
2025-08-16 14:36:40,704 - root - INFO - [Agent] Calling LLM...
2025-08-16 14:36:44,379 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-08-16 14:36:44,380 - root - INFO - [Agent] Assistant:
홍길동님입니다. 스마트스토어 관련해서 도와드릴까요?
2025-08-16 14:36:44,517 - root - INFO - [ChatHistory] Added 2 messages
2025-08-16 14:36:44,517 - root - INFO - Execution time of invoke(): 0m 3.82s


In [5]:
result = agent.invoke(input="영화 추천해줘")

2025-08-16 14:36:44,521 - root - INFO - [Agent] Invoking...
2025-08-16 14:36:44,521 - root - INFO - [Agent] User: 영화 추천해줘
2025-08-16 14:36:44,523 - root - INFO - [ChatHistory] Retrieved 4 messages
2025-08-16 14:36:44,523 - root - INFO - [Agent] Calling LLM...
2025-08-16 14:36:48,444 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-08-16 14:36:48,447 - root - INFO - [Agent] Assistant:
저는 스마트 스토어 FAQ를 위한 챗봇입니다. 스마트 스토어에 대한 질문을 부탁드립니다.
2025-08-16 14:36:48,593 - root - INFO - [ChatHistory] Added 2 messages
2025-08-16 14:36:48,593 - root - INFO - Execution time of invoke(): 0m 4.07s


In [6]:
result = agent.invoke(input="가입시 필요한 서류는 어디로 보내야 하나요?")

2025-08-16 14:36:48,596 - root - INFO - [Agent] Invoking...
2025-08-16 14:36:48,596 - root - INFO - [Agent] User: 가입시 필요한 서류는 어디로 보내야 하나요?
2025-08-16 14:36:48,598 - root - INFO - [ChatHistory] Retrieved 6 messages
2025-08-16 14:36:48,598 - root - INFO - [Agent] Calling LLM...
2025-08-16 14:36:53,130 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-08-16 14:36:53,134 - root - INFO - [Agent] Action: Call tool 'retrieve_faq' with args {'query': '가입시 필요한 서류는 어디로 보내야 하나요? 스마트스토어 가입 서류 제출 방법 서류 제출처'}
2025-08-16 14:36:53,499 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-08-16 14:36:53,534 - root - INFO - [RetrieveFAQ] Retrieved 5 documents
2025-08-16 14:36:53,535 - root - INFO - Execution time of retrieve(): 0m 0.40s
2025-08-16 14:36:53,535 - root - INFO - [Agent] Calling LLM...
2025-08-16 14:37:06,226 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 2

### Stream

In [3]:
full_response = ""
chunk_count = 0
for chunk in agent.stream(input="가입시 필요한 서류는 어디로 보내야 하나요?"):
    full_response += chunk

2025-08-16 14:38:49,499 - root - INFO - Execution time of stream(): 0m 0.00s
2025-08-16 14:38:49,501 - root - INFO - [Agent] Calling LLM...
2025-08-16 14:38:51,633 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-08-16 14:38:51,643 - root - INFO - [Agent] Action: Call tool 'retrieve_faq' with args {'query': '가입시 필요한 서류는 어디로 보내야 하나요?'}
2025-08-16 14:38:52,074 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-08-16 14:38:52,104 - root - INFO - [RetrieveFAQ] Retrieved 5 documents
2025-08-16 14:38:52,105 - root - INFO - Execution time of retrieve(): 0m 0.46s
2025-08-16 14:38:52,105 - root - INFO - [Agent] Streaming response...
2025-08-16 14:38:59,388 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-08-16 14:39:02,255 - root - INFO - [Agent] Assistant:
서류는 별도 우편이나 이메일로 보내는 게 아니라, 가입신청 화면에서 파일로 업로드하시면 됩니다.

- 가입 시: 신청 단계에서 파일 업로드  
- 가입 후 업로드

In [4]:
full_response

'서류는 별도 우편이나 이메일로 보내는 게 아니라, 가입신청 화면에서 파일로 업로드하시면 됩니다.\n\n- 가입 시: 신청 단계에서 파일 업로드  \n- 가입 후 업로드(놓쳤을 경우): 판매자정보 > 심사내역 조회 메뉴의 ‘제출하기’ 버튼으로 업로드 가능  \n- 제출 기한: 가입 신청 후 30일 이내 제출(미제출 시 30일째 자동 ‘보류’, 180일째 유지되면 자동 ‘거부’)  \n- 심사 소요: 제출 완료일 기준 3영업일 이내 심사 진행(결과는 등록된 메일로 안내)  \n\n참고 팁:\n- 휴대폰 촬영 시 서류가 선명하게 보이도록 촬영 후 업로드하세요.  \n- 심사 완료 후에는 판매자정보 > 정보변경 메뉴에서 수정 요청이 가능합니다.  \n\n해외 사업자 등 별도 필요서류가 있는 경우(여권, 사업자증빙, 영문 번역본 등) 추가 안내가 필요하시면 알려주세요.'